<a href="https://colab.research.google.com/github/B-Pearlraj/App-User-Behavior-Segmentation/blob/main/Project_1_EQ_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []

start_year = datetime.now().year - 5
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
    for month in range(1, 13):

        start_date = f"{year}-{month:02d}-01"

        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        params = {
            "format": "geojson",
            "starttime": start_date,
            "endtime": end_date,
            "minmagnitude": 3
        }

        response = requests.get(url, params=params)

        if response.status_code != 200:
            print(f"Failed for {start_date}")
            continue

        data = response.json()

        for f in data["features"]:
            p = f["properties"]
            g = f["geometry"]["coordinates"]

            all_records.append({
                "id": f.get("id"),
                "time": pd.to_datetime(p.get("time"), unit="ms"),
                "updated": pd.to_datetime(p.get("updated"), unit="ms"),

                "latitude": g[1] if len(g) > 1 else None,
                "longitude": g[0] if len(g) > 0 else None,
                "depth_km": g[2] if len(g) > 2 else None,

                "mag": p.get("mag"),
                "magType": p.get("magType"),
                "place": p.get("place"),
                "status": p.get("status"),
                "tsunami": p.get("tsunami"),
                "sig": p.get("sig"),
                "mmi": p.get("mmi"),
                "alert": p.get("alert"),
                "felt": p.get("felt"),
                "cdi": p.get("cdi"),

                "net": p.get("net"),
                "code": p.get("code"),
                "ids": p.get("ids"),
                "sources": p.get("sources"),
                "types": p.get("types"),

                "nst": p.get("nst"),
                "dmin": p.get("dmin"),
                "rms": p.get("rms"),
                "gap": p.get("gap"),
                "type": p.get("type")
            })

# Create DataFrame
df = pd.DataFrame(all_records)

# Display summary
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Show first 5 rows
print(df.head())

# Show column names
print("\nColumns:")
print(df.columns.tolist())


Rows: 113252
Columns: 26
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  net  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   us   
1                                        Fiji region  reviewed  ...   us   
2                    103 km SW of Basco, Philippines  reviewe

In [ ]:
df.columns

Index(['id', 'time', 'updated', 'latitude', 'longitude', 'depth_km', 'mag',
       'magType', 'place', 'status', 'tsunami', 'sig', 'mmi', 'alert', 'felt',
       'cdi', 'net', 'code', 'ids', 'sources', 'types', 'nst', 'dmin', 'rms',
       'gap', 'type'],
      dtype='object')

In [ ]:
df['place']

,place
0,"29 km SW of Villa Basilio Nievas, Argentina"
1,Fiji region
2,"103 km SW of Basco, Philippines"
3,"114 km N of M?n?b, Iran"
4,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen"
...,...
113247,"Izu Islands, Japan region"
113248,southeast of Easter Island
113249,"Izu Islands, Japan region"
113250,"63 km N of Charlotte Amalie, U.S. Virgin Islands"


In [ ]:
import re

In [ ]:
df['country'] = df['place'].apply(lambda x: re.search(r', ([^,]+)$', str(x)).group(1) if re.search(r', ([^,]+)$', str(x)) else str(x).strip())

In [ ]:
df[['place','country']]

,place,country
0,"29 km SW of Villa Basilio Nievas, Argentina",Argentina
1,Fiji region,Fiji region
2,"103 km SW of Basco, Philippines",Philippines
3,"114 km N of M?n?b, Iran",Iran
4,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",Svalbard and Jan Mayen
...,...,...
113247,"Izu Islands, Japan region",Japan region
113248,southeast of Easter Island,southeast of Easter Island
113249,"Izu Islands, Japan region",Japan region
113250,"63 km N of Charlotte Amalie, U.S. Virgin Islands",U.S. Virgin Islands


In [ ]:
df.describe(include='all')

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,code,ids,sources,types,nst,dmin,rms,gap,type,country
count,113252,113252,113252,113252.000000,113252.000000,113252.000000,113252.000000,113252,113252,113252,...,113252,113252,113252,113252,85490.000000,109277.000000,113233.000000,110230.000000,113252,113252
unique,113252,NaN,NaN,NaN,NaN,NaN,NaN,17,62490,2,...,113252,113252,369,580,NaN,NaN,NaN,NaN,11,531
top,us7000spra,NaN,NaN,NaN,NaN,NaN,NaN,mb,South Sandwich Islands region,reviewed,...,7000spra,",us7000spra,",",us,",",origin,phase-data,",NaN,NaN,NaN,NaN,earthquake,Alaska
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,80280,3119,113085,...,1,1,82599,86138,NaN,NaN,NaN,NaN,112391,13821
mean,NaN,2023-09-18 09:30:31.228902912,2024-01-05 10:02:40.733505792,12.103575,7.299030,71.911932,4.256614,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,45.422447,3.186712,0.656981,122.451071,NaN,NaN
min,NaN,2021-01-01 00:14:07.580000,2021-01-01 10:15:32.601000,-84.493200,-179.999700,-3.740000,3.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,8.000000,NaN,NaN
25%,NaN,2022-04-18 02:53:59.148999936,2022-08-09 06:32:14.295750144,-14.711600,-113.942675,10.000000,4.000000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,20.000000,0.829000,0.490000,75.000000,NaN,NaN
50%,NaN,2023-09-23 19:59:09.098500096,2024-01-18 20:38:22.971500032,14.973500,23.911400,21.550000,4.300000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,32.000000,1.809000,0.650000,113.000000,NaN,NaN
75%,NaN,2025-02-26 06:47:32.816250112,2025-06-14 18:26:41.040000,39.071400,130.524200,71.092500,4.600000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,56.000000,3.727000,0.820000,156.000000,NaN,NaN
max,NaN,2026-06-11 05:38:46.440000,2026-06-11 06:53:20.040000,87.375200,179.999400,683.578000,8.800000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,619.000000,62.558000,3.400000,358.180000,NaN,NaN


In [ ]:
df.select_dtypes(include='object').values.T

array([['us6000ddi8', 'us6000dev6', 'us6000dev5', ..., 'us7000spru',
        'pr71518453', 'us7000spra'],
       ['mwr', 'mb', 'mb', ..., 'mww', 'md', 'mb'],
       ['29 km SW of Villa Basilio Nievas, Argentina', 'Fiji region',
        '103 km SW of Basco, Philippines', ...,
        'Izu Islands, Japan region',
        '63 km N of Charlotte Amalie, U.S. Virgin Islands',
        'Macquarie Island region'],
       ...,
       [',dyfi,moment-tensor,origin,phase-data,', ',origin,phase-data,',
        ',origin,phase-data,', ...,
        ',internal-moment-tensor,internal-origin,losspager,moment-tensor,origin,phase-data,shakemap,',
        ',origin,phase-data,', ',origin,phase-data,'],
       ['earthquake', 'earthquake', 'earthquake', ..., 'earthquake',
        'earthquake', 'earthquake'],
       ['Argentina', 'Fiji region', 'Philippines', ..., 'Japan region',
        'U.S. Virgin Islands', 'Macquarie Island region']], dtype=object)

In [ ]:
print(df.head())

           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...      code  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...  6000ddi8   
1                                        Fiji region  reviewed  ...  6000dev6   
2                    103 km SW of Basco, Philippines  reviewed  ...  60

In [ ]:
if 'alert' in df.columns:
    df['alert'] = df['alert'].astype(str).str.lower().replace('nan', pd.NA)
    string_cols = ['magType', 'status', 'type', 'net', 'sources', 'types']
for col in string_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).fillna('').str.strip().str.lower()

In [ ]:
df[string_cols].dtypes

,0
magType,object
status,object
type,object
net,object
sources,object
types,object


In [ ]:
df['alert'].isna().sum()

np.int64(0)

In [ ]:
df.isna().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
df[['nst','dmin', 'rms', 'gap', 'mmi', 'felt', 'cdi']].isna().sum()

,0
nst,0
dmin,3975
rms,19
gap,3022
mmi,103505
felt,95888
cdi,95888


In [ ]:
df['nst'] = pd.to_numeric(df['nst'], errors='coerce')
df['nst'] = df['nst'].fillna(df['nst'].median())
df['mmi'] = pd.to_numeric(df['mmi'], errors='coerce')
df['mmi'] = df['mmi'].fillna(df['mmi'].median())
df['felt'] = pd.to_numeric(df['felt'], errors='coerce')
df['felt'] = df['felt'].fillna(df['felt'].median())
df['cdi'] = pd.to_numeric(df['cdi'], errors='coerce')
df['cdi'] = df['cdi'].fillna(df['cdi'].median())
df['dmin'] = pd.to_numeric(df['dmin'], errors='coerce')
df['dmin'] = df['dmin'].fillna(df['dmin'].median())
df['rms'] = pd.to_numeric(df['rms'], errors='coerce')
df['rms'] = df['rms'].fillna(df['rms'].median())
df['gap'] = pd.to_numeric(df['gap'], errors='coerce')
df['gap'] = df['gap'].fillna(df['gap'].median())

In [ ]:
df[['nst','dmin', 'rms', 'gap', 'mmi', 'felt', 'cdi']].isna().sum()

,0
nst,0
dmin,0
rms,0
gap,0
mmi,0
felt,0
cdi,0


In [ ]:
df.isna().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
df['time']

,time
0,2021-01-31 23:20:49.923
1,2021-01-31 23:08:17.161
2,2021-01-31 22:54:19.760
3,2021-01-31 22:06:00.832
4,2021-01-31 21:51:14.016
...,...
113247,2026-06-01 03:58:03.565
113248,2026-06-01 03:40:11.053
113249,2026-06-01 03:34:30.871
113250,2026-06-01 00:33:28.750


In [ ]:
df['time'].dt.year

,time
0,2021
1,2021
2,2021
3,2021
4,2021
...,...
113247,2026
113248,2026
113249,2026
113250,2026


In [ ]:
df['year'] = df['time'].dt.year

In [ ]:
df['time'].dt.month

,time
0,1
1,1
2,1
3,1
4,1
...,...
113247,6
113248,6
113249,6
113250,6


In [ ]:
df['month'] = df['time'].dt.month

In [ ]:
df['time'].dt.day

,time
0,31
1,31
2,31
3,31
4,31
...,...
113247,1
113248,1
113249,1
113250,1


In [ ]:
df['day'] = df['time'].dt.day

In [ ]:
df['time'].dt.day_name()

,time
0,Sunday
1,Sunday
2,Sunday
3,Sunday
4,Sunday
...,...
113247,Monday
113248,Monday
113249,Monday
113250,Monday


In [ ]:
df['day_of_week'] = df['time'].dt.day_name()

In [ ]:
df.shape

(113252, 31)

In [ ]:
df.isna().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
df['depth_km']

,depth_km
0,17.27
1,426.71
2,46.73
3,10.00
4,10.00
...,...
113247,10.00
113248,10.00
113249,11.00
113250,34.17


In [ ]:
import numpy as np
np.where(df['depth_km'] < 70, 'shallow', 'deep')

array(['shallow', 'deep', 'shallow', ..., 'shallow', 'shallow', 'shallow'],
      dtype='<U7')

In [ ]:
df['depth_category'] = np.where(df['depth_km'] < 70, 'shallow', 'deep')

In [ ]:
df.isna().sum()

,0
id,0
time,0
updated,0
latitude,0
longitude,0
depth_km,0
mag,0
magType,0
place,0
status,0


In [ ]:
df['depth_category']

,depth_category
0,shallow
1,deep
2,shallow
3,shallow
4,shallow
...,...
113247,shallow
113248,shallow
113249,shallow
113250,shallow


In [ ]:
df.shape

(113252, 32)

In [ ]:
df['mag']

,mag
0,4.70
1,4.10
2,4.70
3,4.90
4,4.00
...,...
113247,4.40
113248,5.30
113249,5.70
113250,3.39


In [ ]:
conditions = [(df['mag'] >= 6.0) & (df['mag'] < 7.0),(df['mag'] >= 7.0)]
choices = ['Strong', 'Destructive']
df['strong_destructive_flag'] = np.select(conditions, choices, default='Not Strong/Destructive')


In [ ]:
df['strong_destructive_flag']

,strong_destructive_flag
0,Not Strong/Destructive
1,Not Strong/Destructive
2,Not Strong/Destructive
3,Not Strong/Destructive
4,Not Strong/Destructive
...,...
113247,Not Strong/Destructive
113248,Not Strong/Destructive
113249,Not Strong/Destructive
113250,Not Strong/Destructive


In [ ]:
df.shape

(113252, 33)

In [ ]:
df.to_csv("earthquakes_data.csv", index=False)

In [ ]:
df=pd.read_csv('earthquakes_data.csv')
df.head(10)

,id,time,updated,latitude,longitude,depth_km,mag,magType,place,status,...,rms,gap,type,country,year,month,day,day_of_week,depth_category,strong_destructive_flag
0,us6000ddi8,2021-01-31 23:20:49.923,2021-04-16 19:02:44.040,-31.7493,-68.9337,17.27,4.70,mwr,"29 km SW of Villa Basilio Nievas, Argentina",reviewed,...,0.82,42.0,earthquake,Argentina,2021,1,31,Sunday,shallow,Not Strong/Destructive
1,us6000dev6,2021-01-31 23:08:17.161,2021-04-16 19:03:47.040,-15.4902,-177.2052,426.71,4.10,mb,Fiji region,reviewed,...,0.29,64.0,earthquake,Fiji region,2021,1,31,Sunday,deep,Not Strong/Destructive
2,us6000dev5,2021-01-31 22:54:19.760,2021-04-16 19:03:47.040,19.7529,121.3159,46.73,4.70,mb,"103 km SW of Basco, Philippines",reviewed,...,0.69,106.0,earthquake,Philippines,2021,1,31,Sunday,shallow,Not Strong/Destructive
3,us6000ddhs,2021-01-31 22:06:00.832,2021-04-16 19:02:43.040,28.1524,57.2570,10.00,4.90,mb,"114 km N of M?n?b, Iran",reviewed,...,0.61,71.0,earthquake,Iran,2021,1,31,Sunday,shallow,Not Strong/Destructive
4,us6000dev4,2021-01-31 21:51:14.016,2021-04-16 19:03:46.040,71.3212,-3.7578,10.00,4.00,mb,"184 km ENE of Olonkinbyen, Svalbard and Jan Mayen",reviewed,...,0.50,65.0,earthquake,Svalbard and Jan Mayen,2021,1,31,Sunday,shallow,Not Strong/Destructive
5,pr2021031019,2021-01-31 21:34:54.690,2021-04-16 19:02:43.040,18.9996,-65.4121,46.00,3.55,md,"76 km NNE of Luquillo, Puerto Rico",reviewed,...,0.31,305.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
6,pr2021031018,2021-01-31 21:26:30.180,2021-01-31 21:52:15.880,19.0250,-65.4221,30.00,3.27,md,"78 km NNE of Luquillo, Puerto Rico",reviewed,...,0.37,305.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
7,pr2021031020,2021-01-31 21:14:31.020,2021-04-16 19:02:43.040,19.0436,-65.3590,28.00,3.48,md,"82 km N of Culebra, Puerto Rico",reviewed,...,0.37,309.0,earthquake,Puerto Rico,2021,1,31,Sunday,shallow,Not Strong/Destructive
8,us6000dev3,2021-01-31 20:57:55.804,2021-04-16 19:03:46.040,5.6376,126.7150,18.68,4.30,mb,"99 km SE of Pondaguitan, Philippines",reviewed,...,0.49,139.0,earthquake,Philippines,2021,1,31,Sunday,shallow,Not Strong/Destructive
9,us6000dev2,2021-01-31 20:54:42.248,2021-04-16 19:03:46.040,5.8436,126.5404,75.53,4.20,mb,"69 km SE of Pondaguitan, Philippines",reviewed,...,0.58,196.0,earthquake,Philippines,2021,1,31,Sunday,deep,Not Strong/Destructive


In [ ]:
import re

# 1. Load Data - Already done in previous steps, time and updated are datetime objects.

# 2. Clean Text Fields

# Extract country from 'place'
# This regex tries to capture the last part after a comma and space. If no comma, it takes the whole string.
df['country'] = df['place'].apply(lambda x: re.search(r', ([^,]+)$', str(x)).group(1) if re.search(r', ([^,]+)$', str(x)) else str(x).strip())

# Normalize alert field to lowercase
if 'alert' in df.columns:
    df['alert'] = df['alert'].astype(str).str.lower().replace('nan', pd.NA)

# Ensure all string fields (magType, status, type, net, sources, types) are clean
string_cols = ['magType', 'status', 'type', 'net', 'sources', 'types']
for col in string_cols:
    if col in df.columns:
        # Convert to string, fill NaN with empty string, strip whitespace, convert to lowercase
        df[col] = df[col].astype(str).fillna('').str.strip().str.lower()

# 3. Clean Numeric Fields

# Convert specified fields to numeric. Use errors='coerce' to turn non-numeric values into NaN.
numeric_cols_to_convert = ['mag', 'depth_km', 'nst', 'dmin', 'rms', 'gap', 'sig', 'mmi', 'felt', 'cdi']
for col in numeric_cols_to_convert:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing numeric values with median where appropriate
# Identify numeric columns with missing values
numeric_cols_with_nan = ['mmi', 'felt', 'cdi', 'nst', 'dmin', 'rms', 'gap']

for col in numeric_cols_with_nan:
    if col in df.columns and df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)


print("Data cleaning and transformation complete.")
print("DataFrame Info after cleaning:")
df.info()
print("\nFirst 5 rows of DataFrame after cleaning:")
print(df.head())

In [ ]:
import numpy as np

# 1. Add Year, Month, Day, Day of Week from 'time'
# Ensure 'time' column is datetime type first, though it already is according to df.info()
# df['time'] = pd.to_datetime(df['time'], errors='coerce') # Uncomment if 'time' is not already datetime

df['year'] = df['time'].dt.year
df['month'] = df['time'].dt.month
df['day'] = df['time'].dt.day
df['day_of_week'] = df['time'].dt.day_name()

# 2. Add Shallow/Deep earthquake flag based on depth_km
# Common threshold for shallow vs deep earthquakes is 70 km
df['depth_category'] = np.where(df['depth_km'] < 70, 'shallow', 'deep')

# 3. Add Strong/Destructive flag based on mag thresholds
# Define magnitude thresholds
conditions = [
    (df['mag'] < 5.0),
    (df['mag'] >= 5.0) & (df['mag'] < 6.0),
    (df['mag'] >= 6.0) & (df['mag'] < 7.0),
    (df['mag'] >= 7.0)
]
choices = ['minor', 'moderate', 'strong', 'destructive']
df['magnitude_category'] = np.select(conditions, choices, default='unknown')

print("Derived columns added successfully.")
print("First 5 rows with new columns:")
print(df[['time', 'year', 'month', 'day', 'day_of_week', 'depth_km', 'depth_category', 'mag', 'magnitude_category']].head())